In [1]:
import os
import re
import random
import string
import requests

import numpy as np
import pandas as pd
import geopandas as gpd

import arcpy
import arcpy.sa as sa
import arcpy.mp as mp
arcpy.CheckOutExtension("Spatial")

import time

import shutil

#### Create variables for each part of the Census API URL. This makes it easy to alter the values later.  
#### No spaces allowed!

In [2]:
# Default values
census_base_url = "https://api.census.gov/"
census_dataset_url = "data/2022/acs/acs5"
decennial_dataset_url = "data/2020/dec/pl"
census_variables1 = "GEO_ID,NAME,B01001_001E,B25034_006E,B25034_007E,B25034_008E,B25034_009E,B25034_010E,B25034_011E,B25046_001E,B27010_017E,B27010_033E,B27010_050E,B27010_066E,B28002_013E,C16002_004E,C16002_007E,C16002_010E,C16002_013E,B09019_005E,B09019_008E,B19101_001E,C17002_002E,C17002_003E,C17002_004E,C17002_005E,C17002_006E,C17002_007E,B20005_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_020E,B01001_021E"
census_variables2 = "GEO_ID,B16002_004E,B16002_007E,B16002_010E,B16002_013E,B16002_016E,B16002_019E,B05013_017E,B05013_018E,B05013_019E,B16002_022E,B16002_025E,B01001_017E,B01001_018E,B01001_019E,B01001_016E,B16002_028E,B16002_031E,B16002_034E,B16002_037E,B01001_044E,B01001_045E,B01001_046E,B01001_047E,B01001_048E,B01001_049E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E,B25008_003E"
#census_variables2 = "GEO_ID,B05013_017E,B05013_018E,B05013_019E,B16002_022E,B16002_025E,B01001_017E,B01001_018E,B01001_019E,B01001_016E,B16002_028E,B16002_031E,B16002_034E,B16002_037E,B01001_044E,B01001_045E,B01001_046E,B01001_047E,B01001_048E,B01001_049E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E,B25008_003E"

decennial_variables3 = "GEO_ID,P5_002N,P5_003N,P5_004N,P5_005N,P5_008N,P5_009N,P5_010N"
# Geography is very important to structure correctly. Sometimes if it's incorrect, the data pull still works but
# will generate an error for higher counts of variables (like we're pulling here)
in_geography = r"&for=block%20group:*&in=state:15%20county:*%20tract:*"

#### Combine variables to generate necessary URLs for the API pull:

In [3]:
census_combined_url = f"{census_base_url}{census_dataset_url}"
decennial_combined_url = f"{census_base_url}{decennial_dataset_url}"

cbg_url1 = f"{census_combined_url}?get={census_variables1}{in_geography}"
cbg_url2 = f"{census_combined_url}?get={census_variables2}{in_geography}"
cbg_url3 = f"{decennial_combined_url}?get={decennial_variables3}{in_geography}"

#### Create variables for the Census variable lookup API URLs.  
#### Variable lookup links:  
##### https://api.census.gov/data/2020/dec/pl/variables.html  
##### https://api.census.gov/data/2022/acs/acs5/variables.html  
##### https://api.census.gov/data/2022/acs/acs5/subject/variables.html    
#### Right now we just need the first 3 but may need the commented out  
#### ones if for some additional variables if we add them:

In [4]:
census_fields_url1 = f"{census_combined_url}/variables.json" #detailed variable lookup
census_fields_url2 = f"{census_combined_url}/subject/variables.json" #subject variable lookup with stats
census_fields_url3 = f"{decennial_combined_url}/variables.json"

#census_fields_url7 = f"{census_combined_url}/profile/variables.json" #data profiles lookup for demographic/econ data
#census_fields_url8 = f"{census_combined_url}/cprofile/variables.json" #comparison profiles lookup for 5yr to 5yr comps
#census_fields_url9 = f"{census_combined_url}/spt/variables.json" #supplemental estimates lookup

In [5]:
print("Click link to test URL for first set of variables: " + cbg_url1 + "\r")
print("")
print("Click link to test URL for second set of variables: " + cbg_url2 + "\r")
print("")
print("Click link to test URL for third set of variables (Decennial Census 2020): " + cbg_url3 + "\r")
print("")
print("Click link to test URL for the field/variable list with descriptions: " + census_fields_url1 + "\r")
print("Click link to test URL for the field/variable list with descriptions: " + census_fields_url2 + "\r")
print("Click link to test URL for the field/variable list with descriptions: " + census_fields_url3 + "\r")

Click link to test URL for first set of variables: https://api.census.gov/data/2022/acs/acs5?get=GEO_ID,NAME,B01001_001E,B25034_006E,B25034_007E,B25034_008E,B25034_009E,B25034_010E,B25034_011E,B25046_001E,B27010_017E,B27010_033E,B27010_050E,B27010_066E,B28002_013E,C16002_004E,C16002_007E,C16002_010E,C16002_013E,B09019_005E,B09019_008E,B19101_001E,C17002_002E,C17002_003E,C17002_004E,C17002_005E,C17002_006E,C17002_007E,B20005_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_020E,B01001_021E&for=block%20group:*&in=state:15%20county:*%20tract:*

Click link to test URL for second set of variables: https://api.census.gov/data/2022/acs/acs5?get=GEO_ID,B16002_004E,B16002_007E,B16002_010E,B16002_013E,B16002_016E,B16002_019E,B05013_017E,B05013_018E,B05013_019E,B16002_022E,B16002_025E,B01001_017E,B01001_018E,B01001_019E,B01001_016E,B16002_028E,B16002_031E,B16002_034E,B16002_037E,B01001_044E,B01001_045E,B01001_046E,B01001_047E,B01001_048E,B01001_049E,B02001_002E,B02001_003E,B02001_004E,

#### Pull multiple variable listings from Census API and store each in a dataframe:

In [6]:
fields_dict1 = requests.get(census_fields_url1).json()
fields_df1 = pd.DataFrame.from_dict(fields_dict1["variables"], orient="index").reset_index()
fields_df1.head()

,index,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,B24022_060E,Estimate!!Total:!!Female:!!Service occupations...,Sex by Occupation and Median Earnings in the P...,int,B24022,0,NaN,NaN,"B24022_060EA,B24022_060M,B24022_060MA",NaN
4,B19001B_014E,"Estimate!!Total:!!$100,000 to $124,999",Household Income in the Past 12 Months (in 202...,int,B19001B,0,NaN,NaN,"B19001B_014EA,B19001B_014M,B19001B_014MA",NaN


In [7]:
fields_dict2 = requests.get(census_fields_url2).json()
fields_df2 = pd.DataFrame.from_dict(fields_dict2["variables"], orient="index").reset_index()
fields_df2.head()

,index,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,S0804_C04_068E,Estimate!!Public transportation (excluding tax...,Means of Transportation to Work by Selected Ch...,float,S0804,0,NaN,NaN,"S0804_C04_068EA,S0804_C04_068M,S0804_C04_068MA",NaN
4,S0503_C02_078E,Estimate!!Foreign born; Born in Europe!!Civili...,Selected Characteristics of the Foreign-Born P...,float,S0503,0,NaN,NaN,"S0503_C02_078EA,S0503_C02_078M,S0503_C02_078MA",NaN


In [8]:
fields_dict3 = requests.get(census_fields_url3).json()
fields_df3 = pd.DataFrame.from_dict(fields_dict3["variables"], orient="index").reset_index()
fields_df3.head()

,index,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,P4_003N,!!Total:!!Not Hispanic or Latino:,"HISPANIC OR LATINO, AND NOT HISPANIC OR LATINO...",int,P4,0,NaN,NaN,P4_003NA,NaN
4,P4_015N,!!Total:!!Not Hispanic or Latino:!!Population...,"HISPANIC OR LATINO, AND NOT HISPANIC OR LATINO...",int,P4,0,NaN,NaN,P4_015NA,NaN


#### Concatenate the multiple dataframes to create one big one with all variables:

In [9]:
fields_df = pd.concat([fields_df1, fields_df2, fields_df3], ignore_index=True)
print("Fields aka variables data have been concatenated")

Fields aka variables data have been concatenated


### We have the field list pulled from the API but now we need the actual data  
#### Pull CBG level data from Census API and store in dataframe (first set of variables):

In [10]:
cbg_response = requests.get(cbg_url1)
cbg_data = cbg_response.json()
cbg_df = pd.DataFrame(cbg_data[1:], columns=cbg_data[0])
cbg_df.head()
print("Status (200 is good):")
print(cbg_response.status_code)

Status (200 is good):
200


In [11]:
cbg_response2 = requests.get(cbg_url2)
cbg_data2 = cbg_response2.json()
cbg_df2 = pd.DataFrame(cbg_data2[1:], columns=cbg_data2[0])
cbg_df2.head()
print("Status (200 is good):")
print(cbg_response2.status_code)

Status (200 is good):
200


In [12]:
cbg_response3 = requests.get(cbg_url3)
cbg_data3 = cbg_response3.json()
cbg_df3 = pd.DataFrame(cbg_data3[1:], columns=cbg_data3[0])
cbg_df3.head()
print("Status (200 is good):")
print(cbg_response3.status_code)


Status (200 is good):
200


#### Find duplicate fields in the second DF and remove all except GEO_ID which is needed to join
#### Merge tables on GEO_ID and add suffix to duplicated field (should just be GEO_ID):

In [13]:
duplicate_columns = [col for col in cbg_df2.columns if col in cbg_df.columns and col != 'GEO_ID']

# Drop them from cbg_df2
cbg_df2 = cbg_df2.drop(columns=duplicate_columns)

# Now safe to join but still add a suffix if a field is duplicated
cbg_df = cbg_df.join(cbg_df2, how='left', lsuffix='', rsuffix='_2')

#cbg_df = cbg_df0

cbg_df.head()

,GEO_ID,NAME,B01001_001E,B25034_006E,B25034_007E,B25034_008E,B25034_009E,B25034_010E,B25034_011E,B25046_001E,...,B01001_048E,B01001_049E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E,B25008_003E
0,1500000US150010201001,Block Group 1; Census Tract 201; Hawaii County...,1462,82,294,87,17,5,9,813,...,8,21,178,0,14,450,418,21,381,720
1,1500000US150010201002,Block Group 2; Census Tract 201; Hawaii County...,602,24,7,57,28,29,76,505,...,5,29,222,2,0,125,28,69,156,103
2,1500000US150010201003,Block Group 3; Census Tract 201; Hawaii County...,1339,148,53,70,141,67,49,1399,...,7,29,435,3,0,272,65,1,563,150
3,1500000US150010201004,Block Group 4; Census Tract 201; Hawaii County...,1096,32,48,56,45,24,144,789,...,4,2,146,0,0,310,70,36,534,256
4,1500000US150010202021,Block Group 1; Census Tract 202.02; Hawaii Cou...,1279,43,31,24,35,22,72,561,...,5,0,584,14,1,166,254,22,238,113


#### Find duplicate fields in third dataframe and remove all except GEO_ID which is needed to join
#### Merge tables on GEO_ID and add suffix to duplicated field (should just be GEO_ID):

In [14]:
duplicate_columns = [col for col in cbg_df3.columns if col in cbg_df.columns and col != 'GEO_ID']

# Drop them from cbg_df2
cbg_df3 = cbg_df3.drop(columns=duplicate_columns)

# Now safe to join but still add a suffix if a field is duplicated
cbg_df = cbg_df.join(cbg_df3, how='left', lsuffix='', rsuffix='_2')

cbg_df.head()

,GEO_ID,NAME,B01001_001E,B25034_006E,B25034_007E,B25034_008E,B25034_009E,B25034_010E,B25034_011E,B25046_001E,...,B02001_008E,B25008_003E,GEO_ID_2,P5_002N,P5_003N,P5_004N,P5_005N,P5_008N,P5_009N,P5_010N
0,1500000US150010201001,Block Group 1; Census Tract 201; Hawaii County...,1462,82,294,87,17,5,9,813,...,381,720,1500000US150010201002,0,0,0,0,0,0,0
1,1500000US150010201002,Block Group 2; Census Tract 201; Hawaii County...,602,24,7,57,28,29,76,505,...,156,103,1500000US150010201001,0,0,0,0,0,0,0
2,1500000US150010201003,Block Group 3; Census Tract 201; Hawaii County...,1339,148,53,70,141,67,49,1399,...,563,150,1500000US150010201003,0,0,0,0,0,0,13
3,1500000US150010201004,Block Group 4; Census Tract 201; Hawaii County...,1096,32,48,56,45,24,144,789,...,534,256,1500000US150010201004,0,0,0,0,0,0,0
4,1500000US150010202021,Block Group 1; Census Tract 202.02; Hawaii Cou...,1279,43,31,24,35,22,72,561,...,238,113,1500000US150010202021,320,320,0,0,0,0,0


#### Create path for output folder and store it as a variable:

In [15]:
output_folder_path = os.path.join(os.getcwd(), "output")
print ("Output folder: " + output_folder_path)

Output folder: C:\github\CCSVI\Scripts\Census_Download\output


#### Generate table names from the URL segments so that it's clear what year/data is included in the output table:

In [16]:
#general_table_name = func_clean_table_name(f"{census_dataset_url}")
#fields_table_name = f"{general_table_name}_Variables"
#cbg_table_name = f"{general_table_name}_CBG"

fields_table_name = f"census_variables"
cbg_table_name = f"census_cbg_data"

#### Generate output folder and use path to store CBG data pull as a CSV file

In [17]:
os.makedirs(output_folder_path, exist_ok=True)
cbg_csv_path = os.path.join(output_folder_path, f"{cbg_table_name}.csv")
cbg_df.to_csv(cbg_csv_path, index=False, encoding="utf-8")
print(f"CSV file created successfully: {cbg_csv_path}")

CSV file created successfully: C:\github\CCSVI\Scripts\Census_Download\output\census_cbg_data.csv


#### Generate path to store Census variable list as a CSV file:

In [18]:
fields_csv_path = os.path.join(output_folder_path, f"{fields_table_name}.csv")
fields_df.to_csv(fields_csv_path, index=False, encoding="utf-8")
print(f"CSV file created successfully: {fields_csv_path}")

CSV file created successfully: C:\github\CCSVI\Scripts\Census_Download\output\census_variables.csv


In [19]:
print("Census pull complete! Output files are located at: " + output_folder_path)

Census pull complete! Output files are located at: C:\github\CCSVI\Scripts\Census_Download\output


In [ ]:
# Save the result to a new CSV file
merged_df.to_csv("output/spatial_stats_and_census.csv", index=False)